In [95]:
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import SVC


%load_ext autoreload
%autoreload 2
import importlib
import sys
sys.path.append("../src/")
import modele
importlib.reload(modele)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'modele' from 'c:\\Users\\anton\\Downloads\\A3\\analyse données\\Winunmax\\notebooks\\../src\\modele.py'>

In [96]:
df = pd.read_csv("../data/preparee/dataframe_traite.csv")
#Je laisse game_id pour le moment pour des vérifications
X = df.drop(columns=["game_id","home_club_name","away_club_name","date","season","home_club_id","away_club_id","results","difference classement","home_club_position","away_club_position"]) 
Y = df["results"]
display(X.head(3))
display("classes déséquilibrée ",Y.value_counts())
display(X.info())

,Difference value home et away,Ratio value home par away,LOG Ratio value home par away,winrate_home 1 ans,winrate_away 1 ans,difference winrate home-away 1 ans,winrate_home 3 ans,winrate_away 3 ans,difference winrate home-away 3 ans,evolution difference winrate home-away entre 3 et 1 ans,...,pos_moy_away_1_ans,difference classement moyen home-away 1 ans,pos_moy_home_3_ans,pos_moy_away_3_ans,difference classement moyen home-away 3 ans,evolution classement home 3 et 1 ans,evolution classement away 3 et 1 ans,evolution difference classement entre équipes entre 3 et 1 ans,nouveau club home,nouveau club away
0,17500000.0,2.206897,0.791587,0.618421,0.447368,0.171053,0.618421,0.447368,0.171053,0.0,...,12.500000,-6.026316,6.473684,12.500000,-6.026316,0.0,0.0,0.0,0,0
1,-500000.0,0.928058,-0.074662,0.394737,0.350000,0.044737,0.394737,0.350000,0.044737,0.0,...,20.000000,-3.289474,16.710526,20.000000,-3.289474,0.0,0.0,0.0,0,1
2,-18200000.0,0.257143,-1.358123,0.473684,0.605263,-0.131579,0.473684,0.605263,-0.131579,0.0,...,8.236842,0.815789,9.052632,8.236842,0.815789,0.0,0.0,0.0,0,0


'classes déséquilibrée '

results
 1    1802
-1    1203
 0    1073
Name: count, dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4078 entries, 0 to 4077
Data columns (total 23 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   Difference value home et away                                   4078 non-null   float64
 1   Ratio value home par away                                       4078 non-null   float64
 2   LOG Ratio value home par away                                   4078 non-null   float64
 3   winrate_home 1 ans                                              4078 non-null   float64
 4   winrate_away 1 ans                                              4078 non-null   float64
 5   difference winrate home-away 1 ans                              4078 non-null   float64
 6   winrate_home 3 ans                                              4078 non-null   float64
 7   winrate_away 3 ans                                 

None

In [97]:
mi = mutual_info_classif(X, Y, random_state=42)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print(mi_series)

difference winrate home-away 1 ans                                0.055161
difference winrate home-away 3 ans                                0.053257
pos_moy_home_3_ans                                                0.048952
LOG Ratio value home par away                                     0.042963
evolution classement away 3 et 1 ans                              0.042415
Ratio value home par away                                         0.041955
Difference value home et away                                     0.041636
pos_moy_home_1_ans                                                0.041394
evolution classement home 3 et 1 ans                              0.036595
pos_moy_away_3_ans                                                0.035843
winrate_away 3 ans                                                0.035128
pos_moy_away_1_ans                                                0.033030
winrate_home 1 ans                                                0.031868
winrate_home 3 ans       

In [98]:
#Modèle 1 de référence (NAIF)

y_1_pred = pd.Series([1]*len(X))

modele.affichage_complet_score(Y,y_1_pred)


Accuracy : 0.441883276115743
Precision : 0.14729442537191434
Recall : 0.3333333333333333
F1 Score : 0.20430839002267573
Matrice de confusion :
 [[   0    0 1203]
 [   0    0 1073]
 [   0    0 1802]]


c:\Users\anton\Downloads\A3\analyse données\Winunmax\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [99]:
#Modèle 2 de référence

y_2_pred = modele.prediction_modele_simple_diff_winrate(X["difference winrate home-away 1 ans"])

modele.affichage_complet_score(Y,y_2_pred)


Accuracy : 0.43550760176557135
Precision : 0.4261541192070646
Recall : 0.42621475882213966
F1 Score : 0.4225909126838525
Matrice de confusion :
 [[609 339 255]
 [386 331 356]
 [447 519 836]]


In [100]:
df_sans_2012 = pd.read_csv("../data/preparee/dataframe_traite_sans_2012.csv")

X_sans_2012 = df_sans_2012.drop(columns=["game_id","home_club_name","away_club_name","date","season","home_club_id","away_club_id","results","difference classement","home_club_position","away_club_position"]) 
Y_sans_2012 = df_sans_2012["results"]



In [101]:
mi_sans_2012 = mutual_info_classif(X_sans_2012, Y_sans_2012, random_state=42)
mi_series_sans_2012 = pd.Series(mi_sans_2012, index=X_sans_2012.columns).sort_values(ascending=False)
print("On remarque que les features sont plus informatives et que les features qui se basent sur" \
" l’historique apparaissent \nplus importantes (6/6 premiers, alors que 4/6 en incluant 2012), or on utilisera pour prédire 2023 des features historiques uniquement\n")
print(mi_series_sans_2012)

On remarque que les features sont plus informatives et que les features qui se basent sur l’historique apparaissent 
plus importantes (6/6 premiers, alors que 4/6 en incluant 2012), or on utilisera pour prédire 2023 des features historiques uniquement

difference winrate home-away 1 ans                                0.060935
difference winrate home-away 3 ans                                0.057804
pos_moy_away_3_ans                                                0.057275
pos_moy_away_1_ans                                                0.053728
pos_moy_home_3_ans                                                0.050029
pos_moy_home_1_ans                                                0.049162
Ratio value home par away                                         0.046617
LOG Ratio value home par away                                     0.045423
Difference value home et away                                     0.044664
evolution classement home 3 et 1 ans                              0.0418

In [102]:
#Modèle 2 de référence

y_2_pred_sans_2012 = modele.prediction_modele_simple_diff_winrate(X_sans_2012["difference winrate home-away 1 ans"])

modele.affichage_complet_score(Y_sans_2012,y_2_pred_sans_2012)

print("\nAccuracy meilleure de 1.6 points et F1 score meilleur de 0.2 points, on peut supposer que c’est meilleur mais à vérifier car possiblement du bruit")


Accuracy : 0.45105462412114655
Precision : 0.42524203643208747
Recall : 0.4321588332829049
F1 Score : 0.42446818437999617
Matrice de confusion :
 [[609 237 255]
 [386 223 356]
 [447 349 836]]

Accuracy meilleure de 1.6 points et F1 score meilleur de 0.2 points, on peut supposer que c’est meilleur mais à vérifier car possiblement du bruit


In [103]:
X_train_sans_2012, X_temp_sans_2012, Y_train_sans_2012, Y_temp_sans_2012 = train_test_split(X_sans_2012, Y_sans_2012, test_size=0.3, random_state=42)
X_validation_sans_2012, X_test_sans_2012, Y_validation_sans_2012, Y_test_sans_2012 = train_test_split(X_temp_sans_2012, Y_temp_sans_2012, test_size=0.5, random_state=42)

In [104]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sans_2012_scaled = scaler.fit_transform(X_train_sans_2012)
X_validation_sans_2012_scaled = scaler.fit_transform(X_validation_sans_2012)

In [ ]:
svm_model = SVC(kernel="linear")
svm_model.fit(X_train_sans_2012_scaled, Y_train_sans_2012)
y_svm_prediction_validation_1 = svm_model.predict(X_validation_sans_2012_scaled)
modele.affichage_complet_score(y_svm_prediction_validation_1,Y_validation_sans_2012)


Accuracy : 0.5243243243243243
Precision : 0.43607178340784897
Recall : 0.35515328605874874
F1 Score : 0.3757155213465893
Matrice de confusion :
 [[ 73  34  26]
 [  0   0   0]
 [103 101 218]]


c:\Users\anton\Downloads\A3\analyse données\Winunmax\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [110]:
svm_model = SVC(kernel="linear",class_weight="balanced")
svm_model.fit(X_train_sans_2012_scaled, Y_train_sans_2012)
y_svm_prediction_validation_1 = svm_model.predict(X_validation_sans_2012_scaled)
modele.affichage_complet_score(y_svm_prediction_validation_1,Y_validation_sans_2012)


Accuracy : 0.45765765765765765
Precision : 0.44087229305808534
Recall : 0.44990416292303087
F1 Score : 0.44284031534199336
Matrice de confusion :
 [[ 88  43  37]
 [ 48  43  84]
 [ 40  49 123]]
